# GWAS locus summary + gene annotation + pleiotropy

这个 notebook 做 4 件事：

1. 从每个 phenotype 的 GWAS 结果里筛出候选 SNP  
2. 按相邻物理距离把 SNP 合并成 locus，并取每个 locus 的 top SNP  
3. 用 `KnownCanonicalGene.Drop.hg38.txt` 给 top SNP 做 gene annotation  
4. 跨 phenotype 合并 locus，统计多表型共享位点（pleiotropy）

输出文件默认写到：

- `data/results/gwas_all/summary/all_locus_summary.tsv`
- `data/results/gwas_all/summary/all_locus_summary.filtered.tsv`
- `data/results/gwas_all/summary/pleiotropy_summary.tsv`
- `data/results/gwas_all/summary/pleiotropy_members.tsv`
- `data/results/gwas_all/summary/phenotype_locus_counts.tsv`

默认逻辑：

- 先用 `P < 1e-6` 的 SNP 来定义 locus
- 同染色体上，**相邻 SNP 间距 <= 250 kb** 归为同一个 locus
- 每个 locus 里取最小 P 的 SNP 作为 `top SNP`
- `top_p < 5e-8` 记为 `significant`
- `5e-8 <= top_p < 1e-6` 记为 `suggestive`

如果你后面想改成 500 kb 或者别的阈值，只改参数区即可。

In [ ]:
from pathlib import Path
import os
import re
import glob
import json
import math
from collections import defaultdict

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## Step 1. 参数

In [ ]:
# =========================
# 项目路径
# =========================
PROJECT_ROOT = Path("/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomics_regulation")
RESULT_ROOT = PROJECT_ROOT / "data/results/gwas_all/per_pheno"
SUMMARY_DIR = PROJECT_ROOT / "data/results/gwas_all/summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

# gene annotation 文件
GENE_ANNOTATION_FILE = Path("KnownCanonicalGene.Drop.hg38.txt")
if not GENE_ANNOTATION_FILE.exists():
    alt = PROJECT_ROOT / "data/reference/annotation/KnownCanonicalGene.Drop.hg38.txt"
    if alt.exists():
        GENE_ANNOTATION_FILE = alt

# 优先读 manifest；如果没有就自动扫描
MANIFEST_FILE = SUMMARY_DIR / "plot_manifest.tsv"

# =========================
# locus 定义参数
# =========================
LOCUS_CANDIDATE_P = 1e-6           # 用于先筛 SNP、再划分 locus
GENOME_WIDE_SIG_P = 5e-8           # top SNP 显著阈值
LOCUS_MERGE_GAP_BP = 250_000       # 相邻 SNP 间距 <= 250kb 归同一 locus
CROSS_PHENO_MERGE_GAP_BP = 250_000 # 跨 phenotype 合并共享 locus 的距离阈值
MIN_NSNP_PER_LOCUS = 1             # 若不想保留孤点，可改成 2

# =========================
# 文件扫描参数
# =========================
PREFERRED_SUFFIXES = [
    "*.glm.linear",
    "*.glm.logistic",
    "*.glm.firth",
    "*.assoc",
    "*.tsv",
    "*.tsv.gz",
]

# 若 manifest 有 phenotype/add_file 就直接用；
# 否则从 RESULT_ROOT 递归扫描文件
USE_MANIFEST_FIRST = True

print("PROJECT_ROOT =", PROJECT_ROOT)
print("RESULT_ROOT  =", RESULT_ROOT)
print("SUMMARY_DIR  =", SUMMARY_DIR)
print("GENE_FILE    =", GENE_ANNOTATION_FILE)
print("MANIFEST     =", MANIFEST_FILE)

## Step 2. 读入 gene annotation，并准备 gene mapping 函数

下面保留了你给的 `getgene` 思路，只做了两点小增强：

- 染色体字符串更稳一点
- 当点不在基因内时，返回最近基因，并标注方向  
  - `GENE>`：位点在该基因左边  
  - `<GENE`：位点在该基因右边

In [ ]:
def reformat_chrom(chr_col: pd.Series) -> pd.Series:
    """
    convert chrom column to int:
    1-22, X=23, Y=24, M/MT=25, invalid=-1
    """
    def reformat(x):
        try:
            if pd.isna(x):
                return -1
            if isinstance(x, (int, np.integer)):
                return int(x)
            s = str(x).strip()
            if s.lower().startswith("chr"):
                s = s[3:]
            s = s.upper()
            if s == "X":
                return 23
            if s == "Y":
                return 24
            if s in {"M", "MT"}:
                return 25
            return int(float(s))
        except Exception:
            return -1
    return np.frompyfunc(reformat, 1, 1)(chr_col).astype(int)


known_gene = pd.read_csv(GENE_ANNOTATION_FILE, sep="\t")
known_gene["#hg38.knownCanonical.chrom"] = reformat_chrom(
    known_gene["#hg38.knownCanonical.chrom"]
)

# 去掉明显无用或容易干扰的行
known_gene = known_gene.copy()
if "hg38.kgXref.geneSymbol" in known_gene.columns:
    known_gene = known_gene[known_gene["hg38.kgXref.geneSymbol"].notna()].copy()
    known_gene = known_gene[known_gene["hg38.kgXref.geneSymbol"] != "Y_RNA"].copy()

known_gene = known_gene.sort_values(
    ["#hg38.knownCanonical.chrom", "hg38.knownCanonical.chromStart", "hg38.knownCanonical.chromEnd"]
).reset_index(drop=True)

print("known_gene shape:", known_gene.shape)
known_gene.head()

In [ ]:
def mapped(chrom, pos):
    this_chrom = known_gene[known_gene["#hg38.knownCanonical.chrom"] == chrom]
    bg_b = this_chrom["hg38.knownCanonical.chromStart"] <= pos
    ed_b = this_chrom["hg38.knownCanonical.chromEnd"] >= pos
    hit = this_chrom.loc[bg_b & ed_b, "hg38.kgXref.geneSymbol"]
    if len(hit) > 0:
        return hit.iloc[0]
    return np.nan

uf_mapped = np.frompyfunc(mapped, 2, 1)

def closed(chrom, pos):
    mapped_gene = mapped(chrom, pos)
    if pd.isna(mapped_gene):
        this_chrom = known_gene[known_gene["#hg38.knownCanonical.chrom"] == chrom]
        if len(this_chrom) == 0:
            return np.nan

        s_distance = (this_chrom["hg38.knownCanonical.chromStart"] - pos).abs()
        e_distance = (this_chrom["hg38.knownCanonical.chromEnd"] - pos).abs()

        if s_distance.min() < e_distance.min():
            k_distance = s_distance
            template = "{}>"
        else:
            k_distance = e_distance
            template = "<{}"

        closest = k_distance.idxmin()
        return template.format(this_chrom.loc[closest, "hg38.kgXref.geneSymbol"])
    else:
        return mapped_gene

uf_closed = np.frompyfunc(closed, 2, 1)

def nearest_gene_distance_bp(chrom, pos):
    this_chrom = known_gene[known_gene["#hg38.knownCanonical.chrom"] == chrom]
    if len(this_chrom) == 0:
        return np.nan

    in_gene = (
        (this_chrom["hg38.knownCanonical.chromStart"] <= pos) &
        (this_chrom["hg38.knownCanonical.chromEnd"] >= pos)
    )
    if in_gene.any():
        return 0

    s_distance = (this_chrom["hg38.knownCanonical.chromStart"] - pos).abs()
    e_distance = (this_chrom["hg38.knownCanonical.chromEnd"] - pos).abs()
    return min(s_distance.min(), e_distance.min())

## Step 3. 统一读取 GWAS 结果

这里尽量兼容常见列名：

- 染色体：`#CHROM` / `CHROM` / `CHR` / `chrom`
- 位置：`POS` / `BP` / `pos`
- SNP ID：`ID` / `SNP`
- P 值：`P` / `PVAL` / `p`
- 可选：`REF` / `ALT` / `A1`

输出统一字段：

- `chrom`
- `pos`
- `snp_id`
- `p`
- `variant_key`

In [ ]:
def find_first_existing(cols, candidates):
    upper_map = {str(c).upper(): c for c in cols}
    for x in candidates:
        if x.upper() in upper_map:
            return upper_map[x.upper()]
    return None


def make_variant_key(df, chrom_col, pos_col, ref_col=None, alt_col=None):
    chrom = reformat_chrom(df[chrom_col]).astype(str)
    pos = pd.to_numeric(df[pos_col], errors="coerce").astype("Int64").astype(str)
    if ref_col is not None and alt_col is not None:
        ref = df[ref_col].astype(str)
        alt = df[alt_col].astype(str)
        return chrom + ":" + pos + ":" + ref + ":" + alt
    else:
        snp_col = find_first_existing(df.columns, ["ID", "SNP", "rsID", "RSID"])
        if snp_col is not None:
            return df[snp_col].astype(str)
        return chrom + ":" + pos


def load_one_gwas_result(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"missing file: {path}")

    if path.suffix == ".gz":
        df = pd.read_csv(path, sep="\t", compression="gzip", low_memory=False)
    else:
        df = pd.read_csv(path, sep="\t", low_memory=False)

    chrom_col = find_first_existing(df.columns, ["#CHROM", "CHROM", "CHR", "chrom"])
    pos_col   = find_first_existing(df.columns, ["POS", "BP", "pos"])
    id_col    = find_first_existing(df.columns, ["ID", "SNP", "rsID", "RSID"])
    p_col     = find_first_existing(df.columns, ["P", "PVAL", "PVALUE", "p"])
    ref_col   = find_first_existing(df.columns, ["REF"])
    alt_col   = find_first_existing(df.columns, ["ALT", "A1"])

    missing = {
        "chrom_col": chrom_col,
        "pos_col": pos_col,
        "p_col": p_col,
    }
    missing = [k for k, v in missing.items() if v is None]
    if missing:
        raise ValueError(f"{path.name}: missing required columns -> {missing}; columns={list(df.columns)[:30]}")

    out = pd.DataFrame({
        "chrom": reformat_chrom(df[chrom_col]),
        "pos": pd.to_numeric(df[pos_col], errors="coerce"),
        "p": pd.to_numeric(df[p_col], errors="coerce"),
    })

    if id_col is not None:
        out["snp_id"] = df[id_col].astype(str)
    else:
        out["snp_id"] = (
            out["chrom"].astype(str) + ":" +
            out["pos"].astype("Int64").astype(str)
        )

    out["variant_key"] = make_variant_key(df, chrom_col, pos_col, ref_col, alt_col)

    if ref_col is not None:
        out["ref"] = df[ref_col].astype(str)
    if alt_col is not None:
        out["alt"] = df[alt_col].astype(str)

    # 只保留常规常染色体 + X/Y/MT
    out = out[(out["chrom"] >= 1) & (out["chrom"] <= 25)].copy()
    out = out[out["pos"].notna()].copy()
    out = out[out["p"].notna()].copy()
    out = out[np.isfinite(out["p"])].copy()
    out = out[(out["p"] > 0) & (out["p"] <= 1)].copy()

    out["pos"] = out["pos"].astype(int)
    out = out.sort_values(["chrom", "pos", "p"]).reset_index(drop=True)
    return out

## Step 4. 准备输入文件列表

优先逻辑：

- 如果 `plot_manifest.tsv` 存在，并且里面有 `phenotype` + `add_file` 列，就直接用它
- 否则自动去 `per_pheno` 目录里扫描

In [ ]:
def phenotype_from_path(path: Path) -> str:
    parent = path.parent.name
    stem = path.name

    # 常见情况：per_pheno/Phenotype/Phenotype.xxx.glm.linear
    if parent and parent != ".":
        return parent

    # 兜底
    stem = re.sub(r"\.glm\..*$", "", stem)
    stem = re.sub(r"\.assoc.*$", "", stem)
    stem = re.sub(r"\.tsv(\.gz)?$", "", stem)
    return stem


def build_input_manifest():
    if USE_MANIFEST_FIRST and MANIFEST_FILE.exists():
        mf = pd.read_csv(MANIFEST_FILE, sep="\t")
        if {"phenotype", "add_file"}.issubset(mf.columns):
            use = mf[["phenotype", "add_file"]].copy()
            use["file"] = use["add_file"].astype(str)
            use["exists"] = use["file"].apply(lambda x: Path(x).exists())
            use = use[use["exists"]].copy()
            use = use.drop_duplicates(["phenotype", "file"]).reset_index(drop=True)
            use["source"] = "manifest"
            return use[["phenotype", "file", "source"]]

    files = []
    for pattern in PREFERRED_SUFFIXES:
        files.extend(RESULT_ROOT.rglob(pattern))

    if len(files) == 0:
        raise FileNotFoundError(f"No GWAS result files found under: {RESULT_ROOT}")

    rows = []
    for f in files:
        rows.append({
            "phenotype": phenotype_from_path(f),
            "file": str(f),
            "source": "scan",
        })

    out = pd.DataFrame(rows).drop_duplicates(["phenotype", "file"]).reset_index(drop=True)
    return out


input_manifest = build_input_manifest()
print("n input files:", len(input_manifest))
display(input_manifest.head(20))

## Step 5. 对单个 phenotype 划分 locus

逻辑是：

1. 先取 `p < LOCUS_CANDIDATE_P` 的 SNP  
2. 按 `chrom, pos` 排序  
3. 同染色体上，从前到后遍历  
4. 如果与前一个 SNP 的距离 `<= LOCUS_MERGE_GAP_BP`，归到同一个 locus  
5. 否则开新 locus  
6. 每个 locus 里取最小 `p` 的 top SNP

In [ ]:
def build_locus_table_for_one_pheno(df, phenotype,
                                    locus_candidate_p=LOCUS_CANDIDATE_P,
                                    sig_p=GENOME_WIDE_SIG_P,
                                    merge_gap_bp=LOCUS_MERGE_GAP_BP,
                                    min_nsnp_per_locus=MIN_NSNP_PER_LOCUS):
    cand = df[df["p"] < locus_candidate_p].copy()
    if len(cand) == 0:
        return pd.DataFrame()

    cand = cand.sort_values(["chrom", "pos", "p"]).reset_index(drop=True)

    locus_ids = []
    current_locus = 0
    prev_chrom = None
    prev_pos = None

    for _, row in cand.iterrows():
        chrom = int(row["chrom"])
        pos = int(row["pos"])

        if prev_chrom is None:
            current_locus = 1
        else:
            if (chrom != prev_chrom) or ((pos - prev_pos) > merge_gap_bp):
                current_locus += 1

        locus_ids.append(current_locus)
        prev_chrom = chrom
        prev_pos = pos

    cand["locus_index_in_pheno"] = locus_ids

    out_rows = []
    for locus_idx, sub in cand.groupby("locus_index_in_pheno", sort=True):
        sub = sub.sort_values(["p", "pos"]).reset_index(drop=True)
        top = sub.iloc[0]

        top_gene = closed(int(top["chrom"]), int(top["pos"]))
        mapped_gene = mapped(int(top["chrom"]), int(top["pos"]))
        nearest_dist = nearest_gene_distance_bp(int(top["chrom"]), int(top["pos"]))

        top_p = float(top["p"])
        if top_p < sig_p:
            signal_tier = "significant"
        else:
            signal_tier = "suggestive"

        out_rows.append({
            "phenotype": phenotype,
            "locus_id": f"{phenotype}_L{int(locus_idx):04d}",
            "chrom": int(sub["chrom"].iloc[0]),
            "start": int(sub["pos"].min()),
            "end": int(sub["pos"].max()),
            "locus_width_bp": int(sub["pos"].max() - sub["pos"].min()),
            "nsnp": int(len(sub)),
            "n_sig_p_lt_5e8": int((sub["p"] < sig_p).sum()),
            "n_p_lt_1e6": int((sub["p"] < 1e-6).sum()),
            "top_snp": str(top["snp_id"]),
            "top_variant_key": str(top["variant_key"]),
            "top_pos": int(top["pos"]),
            "top_p": top_p,
            "top_gene": top_gene,
            "mapped_gene_if_inside": mapped_gene,
            "nearest_gene_distance_bp": nearest_dist,
            "signal_tier": signal_tier,
        })

    out = pd.DataFrame(out_rows)
    if min_nsnp_per_locus > 1:
        out = out[out["nsnp"] >= min_nsnp_per_locus].copy()

    out = out.sort_values(["top_p", "chrom", "top_pos"]).reset_index(drop=True)
    return out

## Step 6. 跑全量 phenotype，生成 locus summary

In [ ]:
all_locus_tables = []
failed = []

for i, row in input_manifest.iterrows():
    phenotype = row["phenotype"]
    file = row["file"]

    try:
        gwas = load_one_gwas_result(file)
        locus_df = build_locus_table_for_one_pheno(gwas, phenotype)
        if len(locus_df) > 0:
            locus_df["source_file"] = str(file)
            locus_df["n_variants_total_in_file"] = int(len(gwas))
            all_locus_tables.append(locus_df)
        else:
            failed.append({
                "phenotype": phenotype,
                "file": str(file),
                "status": "no_snp_p_lt_threshold"
            })
    except Exception as e:
        failed.append({
            "phenotype": phenotype,
            "file": str(file),
            "status": f"failed: {repr(e)}"
        })

all_locus_summary = pd.concat(all_locus_tables, ignore_index=True) if len(all_locus_tables) > 0 else pd.DataFrame()
failed_df = pd.DataFrame(failed)

print("all_locus_summary shape:", all_locus_summary.shape)
print("failed shape:", failed_df.shape)

display(all_locus_summary.head(20))
display(failed_df.head(20))

In [ ]:
all_locus_summary = all_locus_summary.sort_values(
    ["top_p", "phenotype", "chrom", "top_pos"]
).reset_index(drop=True)

all_locus_summary_filtered = all_locus_summary.copy()
if MIN_NSNP_PER_LOCUS > 1:
    all_locus_summary_filtered = all_locus_summary_filtered[
        all_locus_summary_filtered["nsnp"] >= MIN_NSNP_PER_LOCUS
    ].copy()

all_locus_summary_file = SUMMARY_DIR / "all_locus_summary.tsv"
all_locus_summary_filtered_file = SUMMARY_DIR / "all_locus_summary.filtered.tsv"
failed_file = SUMMARY_DIR / "all_locus_summary.failed.tsv"

all_locus_summary.to_csv(all_locus_summary_file, sep="\t", index=False)
all_locus_summary_filtered.to_csv(all_locus_summary_filtered_file, sep="\t", index=False)
failed_df.to_csv(failed_file, sep="\t", index=False)

print("saved:", all_locus_summary_file)
print("saved:", all_locus_summary_filtered_file)
print("saved:", failed_file)

## Step 7. phenotype 层面的统计

In [ ]:
if len(all_locus_summary_filtered) > 0:
    phenotype_locus_counts = (
        all_locus_summary_filtered
        .groupby("phenotype", as_index=False)
        .agg(
            n_locus=("locus_id", "nunique"),
            n_significant=("signal_tier", lambda x: int((pd.Series(x) == "significant").sum())),
            best_p=("top_p", "min"),
        )
        .sort_values(["n_significant", "n_locus", "best_p"], ascending=[False, False, True])
        .reset_index(drop=True)
    )
else:
    phenotype_locus_counts = pd.DataFrame()

phenotype_locus_counts_file = SUMMARY_DIR / "phenotype_locus_counts.tsv"
phenotype_locus_counts.to_csv(phenotype_locus_counts_file, sep="\t", index=False)

print("saved:", phenotype_locus_counts_file)
display(phenotype_locus_counts.head(30))

## Step 8. 跨 phenotype 合并 locus，统计 pleiotropy

这里不是按单个 SNP 去数，而是按 **locus 级别** 去合并。

做法：

1. 先把每个 phenotype 的 locus 按 `chrom/start/end` 拿出来  
2. 同一条染色体上，如果两个 locus 的间距 `<= CROSS_PHENO_MERGE_GAP_BP`，就合成同一个 shared locus  
3. 再统计这个 shared locus 覆盖了几个 phenotype

In [ ]:
def merge_loci_across_phenotypes(locus_df, merge_gap_bp=CROSS_PHENO_MERGE_GAP_BP):
    if len(locus_df) == 0:
        return pd.DataFrame(), pd.DataFrame()

    work = locus_df.sort_values(["chrom", "start", "end", "top_p"]).reset_index(drop=True).copy()

    shared_ids = []
    shared_idx = 0
    cur_chr = None
    cur_start = None
    cur_end = None

    for _, row in work.iterrows():
        chrom = int(row["chrom"])
        start = int(row["start"])
        end = int(row["end"])

        if cur_chr is None:
            shared_idx = 1
            cur_chr = chrom
            cur_start = start
            cur_end = end
        else:
            gap = start - cur_end
            if (chrom != cur_chr) or (gap > merge_gap_bp):
                shared_idx += 1
                cur_chr = chrom
                cur_start = start
                cur_end = end
            else:
                cur_end = max(cur_end, end)

        shared_ids.append(shared_idx)

    work["shared_locus_index"] = shared_ids
    work["shared_locus_id"] = work.apply(
        lambda r: f"SL_{int(r['chrom']):02d}_{int(r['shared_locus_index']):05d}",
        axis=1
    )

    pleio_rows = []
    for shared_locus_id, sub in work.groupby("shared_locus_id", sort=True):
        sub = sub.sort_values(["top_p", "phenotype"]).reset_index(drop=True)
        best = sub.iloc[0]

        phenos = sorted(sub["phenotype"].astype(str).unique().tolist())
        top_genes = (
            pd.Series(sub["top_gene"].astype(str))
            .replace("nan", np.nan)
            .dropna()
            .value_counts()
        )
        top_gene = top_genes.index[0] if len(top_genes) > 0 else np.nan

        pleio_rows.append({
            "shared_locus_id": shared_locus_id,
            "chrom": int(sub["chrom"].iloc[0]),
            "start": int(sub["start"].min()),
            "end": int(sub["end"].max()),
            "width_bp": int(sub["end"].max() - sub["start"].min()),
            "n_member_loci": int(len(sub)),
            "n_phenotypes": int(len(phenos)),
            "phenotype_list": ";".join(phenos),
            "best_phenotype": str(best["phenotype"]),
            "best_locus_id": str(best["locus_id"]),
            "best_top_snp": str(best["top_snp"]),
            "best_top_pos": int(best["top_pos"]),
            "best_top_p": float(best["top_p"]),
            "representative_gene": top_gene,
            "best_signal_tier": str(best["signal_tier"]),
        })

    pleio_summary = pd.DataFrame(pleio_rows).sort_values(
        ["n_phenotypes", "best_top_p", "n_member_loci"],
        ascending=[False, True, False]
    ).reset_index(drop=True)

    return work, pleio_summary

In [ ]:
pleiotropy_members, pleiotropy_summary = merge_loci_across_phenotypes(all_locus_summary_filtered)

pleiotropy_members_file = SUMMARY_DIR / "pleiotropy_members.tsv"
pleiotropy_summary_file = SUMMARY_DIR / "pleiotropy_summary.tsv"

pleiotropy_members.to_csv(pleiotropy_members_file, sep="\t", index=False)
pleiotropy_summary.to_csv(pleiotropy_summary_file, sep="\t", index=False)

print("saved:", pleiotropy_members_file)
print("saved:", pleiotropy_summary_file)

display(pleiotropy_summary.head(30))

## Step 9. 导出几个最常用的子表

In [ ]:
sig_locus = all_locus_summary_filtered[
    all_locus_summary_filtered["signal_tier"] == "significant"
].copy().sort_values(["top_p", "nsnp"], ascending=[True, False])

sig_locus_file = SUMMARY_DIR / "significant_locus_summary.tsv"
sig_locus.to_csv(sig_locus_file, sep="\t", index=False)

pleiotropy_top = pleiotropy_summary[
    pleiotropy_summary["n_phenotypes"] >= 2
].copy().sort_values(["n_phenotypes", "best_top_p"], ascending=[False, True])

pleiotropy_top_file = SUMMARY_DIR / "pleiotropy_summary.multi_trait.tsv"
pleiotropy_top.to_csv(pleiotropy_top_file, sep="\t", index=False)

print("saved:", sig_locus_file)
print("saved:", pleiotropy_top_file)

display(sig_locus.head(20))
display(pleiotropy_top.head(20))

## Step 10. 快速 sanity check

这里主要看几件事：

- locus 数量是不是明显比 SNP 数量少很多
- significant locus 是否主要来自少数 phenotype
- pleiotropy 是否有几个特别突出的 shared locus

In [ ]:
print("=== locus summary ===")
print("n phenotype with loci:", all_locus_summary_filtered["phenotype"].nunique() if len(all_locus_summary_filtered) else 0)
print("n loci total:", len(all_locus_summary_filtered))
print("n significant loci:", int((all_locus_summary_filtered["signal_tier"] == "significant").sum()) if len(all_locus_summary_filtered) else 0)
print("n suggestive loci:", int((all_locus_summary_filtered["signal_tier"] == "suggestive").sum()) if len(all_locus_summary_filtered) else 0)

if len(all_locus_summary_filtered):
    print("\n=== top genes ===")
    display(
        all_locus_summary_filtered["top_gene"]
        .astype(str)
        .replace("nan", np.nan)
        .dropna()
        .value_counts()
        .head(20)
        .rename_axis("gene")
        .reset_index(name="n_loci")
    )

if len(pleiotropy_summary):
    print("\n=== top shared loci ===")
    display(
        pleiotropy_summary
        .sort_values(["n_phenotypes", "best_top_p"], ascending=[False, True])
        .head(20)
    )

## Step 11. 给后续画图准备一个候选表

后面如果你要做 regional plot / zoom plot，可以优先画这些：

- `significant`
- `nsnp > 1`
- `n_phenotypes >= 2`
- `best_top_p` 很小

In [ ]:
plot_candidates = (
    pleiotropy_members.merge(
        pleiotropy_summary[["shared_locus_id", "n_phenotypes", "best_top_p"]],
        on="shared_locus_id",
        how="left"
    )
    .sort_values(
        ["n_phenotypes", "top_p", "nsnp"],
        ascending=[False, True, False]
    )
    .reset_index(drop=True)
)

plot_candidates_file = SUMMARY_DIR / "locus_plot_candidates.tsv"
plot_candidates.to_csv(plot_candidates_file, sep="\t", index=False)

print("saved:", plot_candidates_file)
display(plot_candidates.head(30))

## Step 12. 结果文件说明

### 1）`all_locus_summary.tsv`
每个 phenotype 的 locus 总表，包含 suggestive + significant

### 2）`all_locus_summary.filtered.tsv`
过滤后的 locus 总表  
如果 `MIN_NSNP_PER_LOCUS=2`，这里就会自动去掉单点 locus

### 3）`significant_locus_summary.tsv`
只保留 `top_p < 5e-8` 的 locus

### 4）`pleiotropy_members.tsv`
每个 phenotype 的每个 locus，对应到哪个 shared locus

### 5）`pleiotropy_summary.tsv`
shared locus 汇总表，能直接看一个 locus 打了多少 phenotype

### 6）`locus_plot_candidates.tsv`
后续画 zoom plot 的候选列表